# Model Evaluation

In [1]:
# ==== 1) Imports & device ====
import os, json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, roc_auc_score, average_precision_score
)
from sklearn.preprocessing import label_binarize
from datasets import Dataset
from transformers import AutoTokenizer, RobertaForSequenceClassification
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

c:\Users\teomi\Projects\FEDDIE\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [3]:
# ==== 2) Load model & tokenizer ====
# If __file__ is not defined (e.g., in a notebook), replace with a fixed absolute path to /models/...
base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), "..")) if "__file__" in globals() else os.getcwd()

model_dir = os.path.abspath(os.path.join(base_dir, "../models/finetuned_roberta_model_2_pre_overfit_epoch_10_safetensors"))

roberta_tokenizer_pre_overfit = AutoTokenizer.from_pretrained(model_dir)
roberta_model_pre_overfit = RobertaForSequenceClassification.from_pretrained(model_dir, num_labels=3)

roberta_model_pre_overfit = roberta_model_pre_overfit.to(torch.float32).to(device)
roberta_model_pre_overfit.eval()

print("Model & tokenizer loaded from:", model_dir)

Model & tokenizer loaded from: c:\Users\teomi\Projects\FEDDIE\models\finetuned_roberta_model_2_pre_overfit_epoch_10_safetensors


In [4]:
# ==== 3) Load labeled JSON files ====
labeled_data = []

with open(os.path.join(base_dir, "../data/labeled_sample_sentences.json"), "r", encoding="utf-8") as f:
    labeled_data += json.load(f)

with open(os.path.join(base_dir, "../data/news/labeled_cnbc_fed_markets.json"), "r", encoding="utf-8") as f:
    labeled_data += json.load(f)

print(f"Loaded {len(labeled_data)} labeled entries")


Loaded 533 labeled entries


In [5]:
# ==== 4) Flatten sentences & build HF Dataset ====
label_map = {"Hawkish": 0, "Dovish": 1, "Neutral": 2}
sentences, labels = [], []

for entry in labeled_data:
    for sentence in entry["labeled_sentences"]:
        lab = sentence["label"]
        if lab in label_map:
            sentences.append(sentence["sentence"])
            labels.append(label_map[lab])

print(f"Total sentences: {len(sentences)}")
print("Label distribution:", pd.Series(labels).value_counts().sort_index())

dataset = Dataset.from_dict({
    "sentence": sentences,
    "label": labels
})

Total sentences: 6440
Label distribution: 0    1127
1    1427
2    3886
Name: count, dtype: int64


In [6]:
# ==== 5) Tokenize ====
def tokenize_function(example):
    return roberta_tokenizer_pre_overfit(
        example["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128,
        return_attention_mask=True
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 6440/6440 [00:00<00:00, 14759.80 examples/s]


In [7]:
# ==== 6) Convert to tensors (no downsampling) ====
df_full = tokenized_dataset.to_pandas()

print("Full dataset class distribution:")
print(df_full['label'].value_counts().sort_index())

input_ids = np.array(list(df_full["input_ids"]))
attention_mask = np.array(list(df_full["attention_mask"]))
labels_np = np.array(df_full["label"])

input_ids_tensor = torch.tensor(input_ids, dtype=torch.long)
attention_mask_tensor = torch.tensor(attention_mask, dtype=torch.long)
labels_tensor = torch.tensor(labels_np, dtype=torch.long)

print("Final tensor shapes:")
print("Input IDs:", input_ids_tensor.shape)
print("Attention Mask:", attention_mask_tensor.shape)
print("Labels:", labels_tensor.shape)

Full dataset class distribution:
label
0    1127
1    1427
2    3886
Name: count, dtype: int64
Final tensor shapes:
Input IDs: torch.Size([6440, 128])
Attention Mask: torch.Size([6440, 128])
Labels: torch.Size([6440])


In [8]:
# ==== 7) Stratified Train/Val/Test split (70/10/20) ====
indices = np.arange(len(labels_tensor))

# carve out TEST (20%)
trainval_idx, test_idx = train_test_split(
    indices, test_size=0.20, stratify=labels_np, random_state=42
)
# split TRAIN/VAL from remaining (12.5% of trainval -> 10% overall)
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=0.125, stratify=labels_np[trainval_idx], random_state=42
)

def show_dist(name, idx):
    print(f"{name} size: {len(idx)}; class dist:", pd.Series(labels_np[idx]).value_counts().sort_index().to_dict())

show_dist("Train", train_idx)
show_dist("Val", val_idx)
show_dist("Test", test_idx)

Train size: 4508; class dist: {0: 788, 1: 999, 2: 2721}
Val size: 644; class dist: {0: 113, 1: 143, 2: 388}
Test size: 1288; class dist: {0: 226, 1: 285, 2: 777}


In [9]:
# ==== 8) Build DataLoaders ====
full_dataset = TensorDataset(input_ids_tensor, attention_mask_tensor, labels_tensor)

train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=32, shuffle=True)
val_loader   = DataLoader(Subset(full_dataset, val_idx),   batch_size=64, shuffle=False)
test_loader  = DataLoader(Subset(full_dataset, test_idx),  batch_size=64, shuffle=False)

In [10]:
# ==== 9) Eval function ====
@torch.no_grad()
def run_eval(dataloader, model, device):
    all_logits, all_labels = [], []
    model.eval()
    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        input_ids, attention_mask, labels = [t.to(device) for t in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  # [B, C]
        all_logits.append(logits.detach().cpu())
        all_labels.append(labels.detach().cpu())
    logits = torch.cat(all_logits, dim=0).numpy()         # [N, C]
    y_true = torch.cat(all_labels, dim=0).numpy()         # [N]
    y_proba = torch.softmax(torch.tensor(logits), dim=1).numpy()  # [N, C]
    y_pred = y_proba.argmax(axis=1)
    return y_true, y_pred, y_proba

In [11]:
# ==== 10) Evaluate on TEST + metrics ====
label_names = ["Hawkish", "Dovish", "Neutral"]  # 0/1/2

y_true, y_pred, y_proba = run_eval(test_loader, roberta_model_pre_overfit, device)

print("\n=== Classification Report (per-class & macro/micro) ===")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)
prec_micro, rec_micro, f1_micro, _ = precision_recall_fscore_support(
    y_true, y_pred, average="micro", zero_division=0
)
print(f"\nMacro  — P: {prec_macro:.4f}  R: {rec_macro:.4f}  F1: {f1_macro:.4f}")
print(f"Micro  — P: {prec_micro:.4f}  R: {rec_micro:.4f}  F1: {f1_micro:.4f}")

Evaluating:   0%|          | 0/21 [00:00<?, ?it/s]c:\Users\teomi\Projects\FEDDIE\.venv\lib\site-packages\transformers\models\roberta\modeling_roberta.py:357: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
                                                           


=== Classification Report (per-class & macro/micro) ===
              precision    recall  f1-score   support

     Hawkish     0.7000    0.1549    0.2536       226
      Dovish     0.4944    0.6246    0.5519       285
     Neutral     0.8098    0.9151    0.8592       777

    accuracy                         0.7174      1288
   macro avg     0.6681    0.5648    0.5549      1288
weighted avg     0.7208    0.7174    0.6850      1288


Macro  — P: 0.6681  R: 0.5648  F1: 0.5549
Micro  — P: 0.7174  R: 0.7174  F1: 0.7174


In [12]:
# ==== 11) ROC-AUC (one-vs-rest) ====
# Binarize y_true: shape [N, C]
classes = [0, 1, 2]
y_true_bin = label_binarize(y_true, classes=classes)

try:
    auc_macro = roc_auc_score(y_true_bin, y_proba, multi_class="ovr", average="macro")
    auc_weighted = roc_auc_score(y_true_bin, y_proba, multi_class="ovr", average="weighted")
    auc_per_class = roc_auc_score(y_true_bin, y_proba, multi_class="ovr", average=None)
    print(f"ROC-AUC (macro):    {auc_macro:.4f}")
    print(f"ROC-AUC (weighted): {auc_weighted:.4f}")
    for i, cls in enumerate(label_names):
        print(f"ROC-AUC [{cls}]: {auc_per_class[i]:.4f}")
except ValueError as e:
    # Happens if some class is absent in y_true
    print("ROC-AUC could not be computed for some classes:", e)

ROC-AUC (macro):    0.8720
ROC-AUC (weighted): 0.8877
ROC-AUC [Hawkish]: 0.8633
ROC-AUC [Dovish]: 0.8405
ROC-AUC [Neutral]: 0.9121
